In [ ]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2

from google.colab import files

In [ ]:
# Load pretrained Mask R-CNN model

weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
model = maskrcnn_resnet50_fpn(weights=weights)
model.eval()
categories = weights.meta["categories"]
cup_class_id = categories.index("cup")

print("Model loaded.")
print("Target class: cup")
print("Cup class index:", cup_class_id)

In [ ]:
# Upload mug image

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Uploaded image:", image_path)

In [ ]:
# Convert image to tensor

image = Image.open(image_path).convert("RGB")
image_tensor = F.to_tensor(image)

print("Image loaded and converted to tensor.")

In [ ]:
# Generate mug mask

with torch.no_grad():
    prediction = model([image_tensor])[0]

score_threshold = 0.5
best_score = 0
best_mask = None

for i in range(len(prediction["labels"])):
    label_id = prediction["labels"][i].item()
    score = prediction["scores"][i].item()

    if label_id == cup_class_id and score > score_threshold:
        if score > best_score:
            best_score = score
            best_mask = prediction["masks"][i, 0].numpy()

if best_mask is None:
    raise ValueError("No cup was detected. Try another mug image or lower the score threshold.")

print("Cup mask generated.")
print("Confidence score:", round(best_score, 3))

In [ ]:
# Display mask result

mask_threshold = 0.5
binary_mask = (best_mask > mask_threshold).astype(np.uint8)

image_np = np.array(image)
mask_255 = binary_mask * 255

overlay = image_np.copy()
overlay[binary_mask == 1] = overlay[binary_mask == 1] * 0.5 + np.array([255, 0, 0]) * 0.5
overlay = overlay.astype(np.uint8)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_np)
plt.axis("off")
plt.title("Original Image")

plt.subplot(1, 3, 2)
plt.imshow(mask_255, cmap="gray")
plt.axis("off")
plt.title("Mug Mask")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.axis("off")
plt.title("Mask Overlay")

plt.show()

In [ ]:
Save output mask

mask_output_path = "mug_mask.png"
overlay_output_path = "mug_mask_overlay.png"

cv2.imwrite(mask_output_path, mask_255)
cv2.imwrite(overlay_output_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

print("Saved mask:", mask_output_path)
print("Saved overlay:", overlay_output_path)